In [ ]:
!pip install torchsummary torchview

In [ ]:
import os
import numpy as np
import cv2
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import seaborn as sns
import shutil
import time
import pandas as pd
from torchvision import transforms
from torchvision.transforms import v2
from sklearn.metrics import accuracy_score
from matplotlib import pyplot as plt
from torch.utils.data import Dataset,DataLoader,random_split
from torchsummary import summary
from torchview import draw_graph
from warnings import filterwarnings
from tqdm import tqdm
filterwarnings(action='ignore')



In [ ]:
data_path='/kaggle/input/datasets/hau100416/music-mel-spectrogram-dataset'
image_paths=[]
for folder in tqdm(os.listdir(data_path)):
    sub_folder=os.path.join(data_path,folder,'comp_pngs')
    for file_name in os.listdir(sub_folder):
        image_paths.append(os.path.join(sub_folder,file_name))


In [ ]:
def get_train_set_size(image_paths,k=200000):
    return image_paths[:k]

In [ ]:
image_paths=get_train_set_size(image_paths,k=200000)

In [ ]:
def move_to(obj, device):
    if isinstance(obj, list):
        return [move_to(x,device) for x in obj]
    elif isinstance(obj,tuple):
        return tuple(move_to(obj,device))
    elif isinstance(obj,set):
        return set(move_to(obj,device))
    elif isinstance(obj,dict):
        to_return=dict()
        for key,value in obj.items():
            to_return[move_to(key,device)]=move_to(value,device)
        return to_return
    elif hasattr(obj,'to'):
        return obj.to(device)
    return obj

In [ ]:
def run_epoch(model,optimizer,data_loader,loss_func,device,results,score_funcs,prefix="",desc=None,keep=False,epoch=None):
    running_loss=[]
    y_true=[]
    y_pred=[]
    start=time.time()
    for inputs,labels in tqdm(data_loader,desc=desc,leave=keep):
        inputs=move_to(inputs,device)
        labels=move_to(labels,device)
        y_hat,mu,sigma=model(inputs)
        #compute loss.
        reconstructed_loss=loss_func(y_hat,labels)
        kl_ele=1+sigma-mu.pow(2)-sigma.exp()
        kl_div=-0.5*torch.sum(kl_ele)
        # kdl_ele=mu.pow(2).add_(sigma.exp()).mul_(-1).add_(1).add_(sigma)
        # kl_div=torch.sum(kdl_ele).mul_(-0.5)
        loss=reconstructed_loss+kl_div
        if model.training:
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        running_loss.append(loss.item())
        if len(score_funcs) >0 and isinstance(labels,torch.Tensor):
            #moving labels and predictions back to cpu for computing /storing predictions
            labels=labels.detach().cpu().numpy()
            y_hat=y_hat.detach().cpu().numpy()
            y_true.extend(labels.tolist())
    end=time.time()
    y_pred=np.asarray(y_pred)
    if len(y_pred.shape)==2 and y_pred.shape[1]>1:
        #using for classification tasks
        y_pred=np.argmax(y_pred,axis=1)
    results[prefix+' loss'].append(np.mean(running_loss))
    result_str=[f'{prefix} loss: {np.mean(running_loss)}']
    for name,score_func in score_funcs.items():
        try:
            score=score_func(y_true,y_pred)
            results[prefix+' '+name].append(score)
            result_str.append(f'{prefix} {name}: {score}')
            if prefix=='val' or prefix =='test':
                checkpointer(score,(epoch+1),model,optimizer)
        except:
            results[prefix+' '+name].append(float('NaN'))
            result_str.append(f'{prefix} {name}: {float('NaN')}')
    print(' '.join(result_str))
    return end-start

In [ ]:
def train_loop(model,loss_func,train_loader,test_loader=None,val_loader=None,score_funcs=None,epochs=50,device='cpu',
              optimizer=None,lr_schedule=None,checkpoint_file=None,keep=True):
    """full training loop to train a neural network from scratch. this will run through a
    specific number of epochs and calculate scores by using given score functions"""
    if score_funcs is None:
        score_funcs={}
    to_track=['epoch','total time','train loss','lr']
    if val_loader is not None:
        to_track.append('val loss')
    if test_loader is not None:
        to_track.append('test loss')
    for eval_score in score_funcs:
        to_track.append('train ' + eval_score)
        if val_loader is not None:
            to_track.append('val '+eval_score)
        if test_loader is not None:
            to_track.append('test '+eval_score)
    total_train_time=0
    results={'train loss':[],'val loss':[],'total time':[],'epoch':[],'lr':[]}
    for item in to_track:
        results[item]=[]
    if optimizer is None:
        optimizer=torch.optim.AdamW(model.paramerters(),lr=0.001)
        del_opt=True
    else:
        del_opt=False
    model.to(device)
    for epoch in tqdm(range(epochs),desc='Epoch',leave=keep):
        model=model.train()
        run_epoch(model,optimizer,train_loader,loss_func,device,results,score_funcs,prefix='train',desc='Training')
        results['total time'].append(total_train_time)
        results['epoch'].append(epoch)
        results['lr'].append(optimizer.param_groups[0]['lr'])
        if val_loader is not None:
            model=model.eval()
            with torch.no_grad():
                run_epoch(model,optimizer,val_loader,loss_func,device,results,score_funcs,prefix='val',
                         desc='Validation',epoch=epoch)
        if test_loader is not None:
            model=model.eval()
            with torch.no_grad():
                run_epoch(model,optimizer,test_loader, loss_func, device, results, score_funcs, prefix='test',desc='Testing',epoch=epoch)
        if lr_schedule is not None:
            if isinstance(lr_schedule,torch.optim.lr_scheduler.ReduceLROnPlateau):
                lr_schedule.step(results['val loss'][-1])
            else:
                lr_schedule.step()
        if checkpoint_file is not None:
            torch.save({
                'epoch':epoch,
                'model_state_dict':model_state_dict(),
                'optimizer_state_dict':optimizer.state_dict(),
                'results':results
            },checkpoint_file)
    if del_opt:
        del optimizer
    return pd.DataFrame.from_dict(results)

In [ ]:
class AutoEncodeDataset(Dataset):
    """loading input as output for x,x tuples in training data"""
    def __init__(self,img_dir, transform=None):
        self.img_labels=[0]*len(img_dir)
        self.img_dir=img_dir
        self.transform=transform
        self.resize=v2.Compose([v2.Resize(size=27),transforms.ToTensor()])
    def __len__(self):
        return len(self.img_labels)
    def __getitem__(self,idx):
        img_path=self.img_dir[idx]
        image=torchvision.io.read_image(img_path).to(torch.float32)/255
        if self.transform is not None:
            image=transforms.ToPILImage()(image)
            image=self.transform(image)
        # label=transforms.ToPILImage()(image)
        # label=self.resize(label)
        return image,image
        

In [ ]:
#data augmentation steps to use inside dataladers
train_transform=v2.Compose([v2.RandomAffine(degrees=5,translate=(0.05,0.05),scale=(0.98,1.02)),v2.RandomPerspective(distortion_scale=0.5,p=0.5),transforms.ToTensor()])

In [ ]:
training_data=AutoEncodeDataset(image_paths,transform=train_transform)

In [ ]:
train_size=int(len(training_data)*0.8)
test_size=len(training_data)-train_size
train_dataset,test_dataset=random_split(training_data,(train_size,test_size))
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [ ]:
# use cuda if it's available
if torch.cuda.is_available():
    device=torch.device('cuda')
else:
    device=torch.device('cpu')

Variational Autoencoder (VAE)

In [ ]:
class Encode(nn.Module):
    def __init__(self,channels,embeddings, kernel):
        super(Encode,self).__init__()
        self.relu=nn.ReLU()
        self.filters=[32,64,128,256,512]
        self.conv1=nn.Conv2d(channels,self.filters[0],kernel_size=kernel)
        self.bn1=nn.BatchNorm2d(self.filters[0])
        self.conv2=nn.Conv2d(self.filters[0],self.filters[1],kernel_size=kernel,stride=2)
        self.bn2=nn.BatchNorm2d(self.filters[1])
        self.conv3=nn.Conv2d(self.filters[1],self.filters[2],kernel_size=kernel)
        self.bn3=nn.BatchNorm2d(self.filters[2])
        self.conv4=nn.Conv2d(self.filters[2],self.filters[3],kernel_size=kernel,stride=2)
        self.bn4=nn.BatchNorm2d(self.filters[3])
        self.conv5=nn.Conv2d(self.filters[3],self.filters[4],kernel_size=kernel)
        self.bn5=nn.BatchNorm2d(self.filters[4])
        self.pool=nn.AdaptiveAvgPool2d((2,2))
        self.mu=nn.Linear(self.filters[4]*4,embeddings)
        self.sigma=nn.Linear(self.filters[4]*4,embeddings)
    def forward(self,x):
        x=self.bn1(self.relu(self.conv1(x)))
        x=self.bn2(self.relu(self.conv2(x)))
        x=self.bn3(self.relu(self.conv3(x)))
        x=self.bn4(self.relu(self.conv4(x)))
        x=self.bn5(self.relu(self.conv5(x)))
        x=self.pool(x)
        x=torch.flatten(x,start_dim=1)
        return self.mu(x),self.sigma(x)
class Decode(nn.Module):
    def __init__(self,channels,embeddings,kernel):
        super(Decode,self).__init__()
        self.relu=nn.ReLU()
        self.filters=[512,256,128,64,32]
        self.decode_in=nn.Linear(embeddings,self.filters[0]*4)
        self.deconv1=nn.ConvTranspose2d(self.filters[0],self.filters[1],kernel_size=kernel,stride=2,dilation=1,output_padding=1,padding=1)
        self.bn1=nn.BatchNorm2d(self.filters[1])
        self.deconv2=nn.ConvTranspose2d(self.filters[1],self.filters[2],kernel_size=kernel,stride=2,dilation=1,output_padding=1,padding=1)
        self.bn2=nn.BatchNorm2d(self.filters[2])
        self.deconv3=nn.ConvTranspose2d(self.filters[2],self.filters[3],kernel_size=kernel,stride=2,dilation=1,output_padding=1,padding=1)
        self.bn3=nn.BatchNorm2d(self.filters[3])
        self.deconv4=nn.ConvTranspose2d(self.filters[3],self.filters[4],kernel_size=kernel,stride=2,dilation=1,output_padding=1,padding=1)
        self.bn4=nn.BatchNorm2d(self.filters[4])
        self.deconv5=nn.ConvTranspose2d(self.filters[4],self.filters[4],kernel_size=kernel,stride=2,dilation=1,output_padding=1,padding=1)
        self.bn5=nn.BatchNorm2d(self.filters[4])
        self.deconv6=nn.ConvTranspose2d(self.filters[4],self.filters[4],kernel_size=kernel,stride=2,dilation=1,output_padding=1,padding=1)
        self.bn6=nn.BatchNorm2d(self.filters[4])
        self.decode_out=nn.Conv2d(self.filters[4],channels,kernel_size=kernel,padding=1)
        self.upsample=nn.Upsample(size=(128,512),mode='bilinear',align_corners=False)
        
        
    def forward(self,x):
        x=self.decode_in(x)
        x=x.view(-1,512,2,2)
        x=self.bn1(self.relu(self.deconv1(x)))
        x=self.bn2(self.relu(self.deconv2(x)))
        x=self.bn3(self.relu(self.deconv3(x)))
        x=self.bn4(self.relu(self.deconv4(x)))
        x=self.bn5(self.relu(self.deconv5(x)))
        x=self.bn6(self.relu(self.deconv6(x)))
        x=self.decode_out(x)
        x=self.upsample(x)
        return x
class ConvVariationalAutoEncoder(nn.Module):
    """variational autoencoder model"""
    def __init__(self,channels,embeddings,kernel):
        super(ConvVariationalAutoEncoder,self).__init__()
        self.encoder=Encode(channels,embeddings,kernel)
        self.decoder=Decode(channels,embeddings,kernel)
    def forward(self,x):
        mu,sigma=self.encoder(x)
        std=torch.exp(0.5*sigma)
        epsilon=torch.randn_like(std)
        z=epsilon.mul(std).add_(mu)
        return self.decoder(z),mu,sigma

In [ ]:
model=ConvVariationalAutoEncoder(3,256,3)

In [ ]:
summary(model.cuda(),input_size=(3,128,512),batch_size=-1)

model architecture

In [ ]:
model_graph=draw_graph(model.cuda(),input_size=(3,3,28,28),expand_nested=True)
model_graph.visual_graph

MSE LOSS

In [ ]:
loss_func=nn.MSELoss(reduction='sum')

In [ ]:
images,labels=next(iter(train_dataloader))
fix,axes=plt.subplots(nrows=4,ncols=4,figsize=[13,14],dpi=100)
axes=axes.ravel()
for i,(img,label) in enumerate(zip(images,labels)):
    axes[i].imshow(img.numpy()[0],cmap=plt.cm.gray)
    if i>=15:
        break
plt.show()

In [ ]:
labels[0].shape

Training model

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=0.001)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.1)

In [ ]:
# results=train_loop(model,loss_func,train_loader=train_dataloader,
#                   val_loader=test_dataloader,
#                   optimizer=optimizer,
#                   lr_schedule=scheduler,
#                   device=device,
#                   epochs=10)

In [ ]:
# plt.figure(figsize=[12, 6], dpi=200)
# sns.lineplot(x='epoch', y='train loss', data=results, label='train')
# sns.lineplot(x='epoch', y='val loss', data=results, label='validation')
# plt.ylabel('binary cross entropy loss')
# plt.show()

In [ ]:
# model = model.to('cpu')


# def get_digits(dataset):
#     idx = torch.randint(len(dataset), (1,))
    
#     return dataset[idx][0].unsqueeze(0)


# def get_encodings(image):
#     with torch.no_grad():
#         mu, sigma = model.encoder(image)

#     return mu, sigma

# def get_reconstructed(image):
#     with torch.no_grad():
#         recon, _, _ = model(image)
        
#     return recon


# def inference(*, dataset, n_examples=1):
#     out = []
#     image = get_digits(dataset)
#     mu, sigma = get_encodings(image)
#     recon_img = get_reconstructed(image)

#     for example in range(n_examples):
#         epsilon = torch.rand_like(sigma)
#         z = mu + sigma * epsilon

#         out.append(model.decoder(z))
        
#     return image, out, recon_img

In [ ]:
# input_digit, outs, recon_img = inference(dataset=test_dataset, n_examples=10)

# fig, axes = plt.subplots(nrows=2, ncols=6, layout='constrained', figsize=(14, 4))
# gridspec = axes[0, 0].get_subplotspec().get_gridspec()

# for a in axes[:, 0]:
#     a.remove()
    
# for i, a in enumerate(axes[:, 1:].flat):
#     a.axis('off')
#     a.imshow(outs[i].detach().numpy()[0][0])
    
# subfig = fig.add_subfigure(gridspec[:, 0])
# axsLeft = subfig.subplots(1, 2, sharey=True)

# axsLeft[0].imshow(input_digit[0][0])
# axsLeft[0].set_title('original', fontsize=12)
# axsLeft[1].imshow(recon_img[0][0])
# axsLeft[1].set_title('reconstructed', fontsize=12)

# axsLeft[0].axis('off')
# axsLeft[1].axis('off')

# subfig.suptitle('encoder-decoder', fontsize='x-large')
# fig.suptitle('generated samples', fontsize='xx-large')

In [ ]:
# PATH = '/kaggle/working/vae.pth'
# torch.save(model, PATH)

In [ ]:
model=torch.load('/kaggle/input/notebooks/hau100416/vae-for-music-recommendation/vae.pth',weights_only=False)
model.eval()
model.to('cpu')

In [ ]:
def get_embedding(img_path):
    image=cv2.imread(img_path)
    image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image=image/255.0
    image=torch.tensor(image,dtype=torch.float32)
    image=image.permute(2,0,1)
    image=image.unsqueeze(0)
    image=image.to('cpu')
    model.eval()
    with torch.no_grad():
        mu,sigma=model.encoder(image)
    embedding=mu.squeeze(0)
    return embedding

In [ ]:
def build_csv(folder_path):
    results=[]
    for sub_folder in tqdm(os.listdir(folder_path)):
        path=os.path.join(folder_path,sub_folder,'comp_pngs')
        for filename in os.listdir(path):
            result={
                    'id':filename.split('.')[0],
                    'filename':os.path.join(path,filename)
                }
            latent_space=get_embedding(os.path.join(path,filename))
            for idx ,col in enumerate(latent_space):
                    k=f'latent_{idx}'
                    result[k]=float(col)
            results.append(result)
    results_df=pd.DataFrame(results)
    results_df.to_csv('result.csv')
            

In [ ]:
build_csv('/kaggle/input/datasets/hau100416/music-mel-spectrogram-dataset')